## Importações

In [2]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import seaborn as sns
import matplotlib.pyplot as plt

## Carregar dataset

In [4]:
df = pd.read_csv("data/Obesity.csv")

## Remocao da altura e peso

In [6]:
# df['BMI'] = df['Weight'] / (df['Height']**2)
df = df.drop(columns=["Height", "Weight"])

## Separação treino & teste

In [8]:
X = df.drop('Obesity', axis=1)
y = df['Obesity']

## Identificação das Colunas

In [10]:
categoricas = X.select_dtypes(
    include='object'
).columns.tolist()

numericas = X.select_dtypes(
    exclude='object'
).columns.tolist()

print(categoricas)
print(numericas)

['Gender', 'family_history', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']
['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


## Pré-processamento

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numericas
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            categoricas
        )
    ]
)
y = LabelEncoder().fit_transform(y)

## Split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## Modelos

In [16]:
modelos = {
    "Logistic Regression":
        LogisticRegression(max_iter=5000),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric='mlogloss'
        )
}

## Comparação dos modelos

In [18]:
resultados = {}
pipelines = {}

for nome, modelo in modelos.items():

    pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', modelo)
    ])

    pipeline.fit(
        X_train,
        y_train
    )

    pred = pipeline.predict(X_test)
    acc = accuracy_score(
        y_test,
        pred
    )

    resultados[nome] = acc
    pipelines[nome] = pipeline

    print(nome, round(acc,4))

Logistic Regression 0.6241
Random Forest 0.8534
XGBoost 0.844


## Melhor Modelo

In [20]:
resultado_df = pd.DataFrame(
    resultados.items(),
    columns=['Modelo','Accuracy']
)

resultado_df.sort_values(
    by='Accuracy',
    ascending=False,
    inplace=True
)

resultado_df

,Modelo,Accuracy
1,Random Forest,0.853428
2,XGBoost,0.843972
0,Logistic Regression,0.624113


## Avaliação final

In [22]:
melhor_modelo = resultado_df.iloc[0]['Modelo']
best_pipeline = pipelines[melhor_modelo]
pred = best_pipeline.predict(X_test)

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.92      0.89      0.91        54
           1       0.66      0.79      0.72        58
           2       0.83      0.86      0.85        70
           3       0.93      0.92      0.92        60
           4       1.00      0.98      0.99        65
           5       0.79      0.76      0.77        58
           6       0.88      0.76      0.81        58

    accuracy                           0.85       423
   macro avg       0.86      0.85      0.85       423
weighted avg       0.86      0.85      0.86       423



## Exportar modelo

In [24]:
joblib.dump(
    best_pipeline,
    "obesity_model.pkl"
)

['obesity_model.pkl']